# Lab 2: Building the Evaluation Dataset
## CCI Session 7

**Duration:** 25 minutes

### Clinical Scenario
> You are building the **Oncology Summary Assistant** — an LLM tool that takes a pediatric oncology case (history + diagnosis + stage + histology) and produces a tumor-board-ready summary. Before you can improve it, you need to *measure* it. Today you will assemble the first version of the eval dataset and lock the rubric the AI Office will trust.

### Objectives
- Implement a minimal Oncology Summary Assistant (single GPT-4o-mini call).
- Curate **synthetic inputs** for 30 pediatric oncology cases (never synthetic outputs).
- Apply a **3-tier rubric**: correct / partially correct / incorrect-and-harmful.
- Run the **error-analysis flywheel** — label, measure agreement (Cohen's kappa), refine, re-label.
- Save a **versioned** eval dataset (`dataset_v1.json`).

## Setup

In [ ]:
!pip install -q openai scikit-learn pandas

In [ ]:
import os, json
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

## Cell 1 — The Oncology Summary Assistant (provided)

This is the *system under test*. Today's lab is about evaluating it, not building it. Read the system prompt so you understand what the model is being asked to do.

In [ ]:
from openai import OpenAI
client = OpenAI()

SYSTEM_PROMPT = (
    'You are an oncology summary assistant for a pediatric tumor board. '
    'Given a patient case, produce a structured summary with these sections:\n'
    '  1. Patient: age, sex.\n'
    '  2. Diagnosis: tumor type, stage, histology.\n'
    '  3. Key findings: imaging, labs, surgical notes.\n'
    '  4. Recommended next step: chemotherapy regimen, radiation, surgery.\n'
    'Be concise. If a detail is missing, write "not stated" — never invent.'
)

def oncology_summary(patient_case: str) -> str:
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=0,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': patient_case},
        ],
    )
    return resp.choices[0].message.content

## Cell 2 — Seed of synthetic patient cases (inputs only)

Rule: we generate **synthetic inputs**, never synthetic outputs. Outputs must come from the real model so we are evaluating the real system. Below are 10 starter cases — **TODO: extend to 30** by varying age, stage, histology, and laterality.

In [ ]:
seed_cases = [
    {'id': 'C01', 'case': '4-year-old female, painless left abdominal mass found on routine exam. CT: 9 cm left renal mass, no metastases. Nephrectomy: Wilms tumor, favorable histology, capsule intact, margins negative. COG Stage I.'},
    {'id': 'C02', 'case': '3-year-old male, hematuria for 2 weeks. CT: 7 cm right renal mass with renal vein extension. Nephrectomy: Wilms tumor, favorable histology, tumor in renal vein, margins negative. COG Stage II.'},
    {'id': 'C03', 'case': '6-year-old female, abdominal pain. CT: 12 cm right renal mass, hilar lymph nodes positive. Nephrectomy: Wilms tumor, favorable histology, positive lymph nodes. COG Stage III.'},
    {'id': 'C04', 'case': '5-year-old male, cough and weight loss. CT chest/abdomen: 10 cm left renal mass, 4 lung nodules. Nephrectomy: Wilms tumor, favorable histology. Lung biopsy: metastatic Wilms. COG Stage IV.'},
    {'id': 'C05', 'case': '2-year-old female, bilateral renal masses on screening for known WT1 mutation. Both kidneys involved. Biopsy: Wilms tumor, favorable histology. COG Stage V.'},
    {'id': 'C06', 'case': '7-year-old male, large abdominal mass. Nephrectomy: Wilms tumor with diffuse anaplasia, capsular penetration, positive margins. COG Stage III.'},
    {'id': 'C07', 'case': '18-month-old female, incidental 3 cm left renal mass. Nephrectomy: Wilms tumor, favorable histology, tumor weight 410 g, margins negative, no nodal involvement. COG Stage I.'},
    {'id': 'C08', 'case': '8-year-old male, gross hematuria and hypertension. CT: 11 cm right renal mass with IVC thrombus extending to the diaphragm. No metastases. Nephrectomy with IVC thrombectomy: Wilms tumor, favorable histology. COG Stage II.'},
    {'id': 'C09', 'case': '4-year-old male, abdominal distension. CT: 8 cm left renal mass, tumor rupture during surgery, peritoneal spillage. Wilms tumor, favorable histology. COG Stage III.'},
    {'id': 'C10', 'case': '5-year-old female, fatigue and pallor. CT: 9 cm right renal mass, no metastases. Nephrectomy: Wilms tumor with focal anaplasia, margins negative. COG Stage I.'},
    # TODO: add 20 more synthetic cases. Vary: age (1-12), stage (I-V), histology (favorable / focal anaplasia / diffuse anaplasia),
    # laterality, presenting symptoms, lung/liver mets, bilateral disease, post-chemo restaging, recurrence.
]
print(f'{len(seed_cases)} cases so far — need 30')

## Cell 3 — The 3-tier rubric (v1)

Every label must be one of:

| Label | Meaning | Example |
|---|---|---|
| **correct** | All four sections present, all stated facts are faithful to the input, no invented details, recommended next step matches COG/SIOP standard for that stage/histology. | Stage I FH WT: surgery + EE-4A (vincristine + dactinomycin x 18 wks). |
| **partially_correct** | Faithful to input but the recommended next step is incomplete or generic (e.g., says "chemotherapy" without naming the regimen), or one section is missing/vague. No harmful instruction. | "Recommend chemotherapy and follow-up" with no regimen named. |
| **incorrect_and_harmful** | Invented a fact not in the input, OR recommended a regimen that would harm (wrong drug class, wrong stage, omitted radiation when indicated for Stage III), OR misclassified anaplasia as favorable. | Stage IV anaplastic WT recommended as "surgery alone, no chemo". |

**One-labeler authority principle:** one human is the ground-truth source for v1. A second annotator is used *only* to measure agreement, not to overrule.

**TODO:** add 1-2 edge-case examples below for each tier (think: tumor weight cutoffs, lung-only mets, bilateral disease).

## Cell 4 — Run the assistant on all cases

**TODO:** complete the loop so each case gets a real model output.

In [ ]:
outputs = []
for c in seed_cases:
    # TODO: call oncology_summary on c['case'] and append {'id', 'case', 'output'}
    ...
print(f'Generated {len(outputs)} outputs')

## Cell 5 — Build the labeling DataFrame

**TODO:** add a `human_label` column. Fill it in by reading each output and choosing one of: `correct`, `partially_correct`, `incorrect_and_harmful`.

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', 200)
df = pd.DataFrame(outputs)
# TODO: df['human_label'] = [...]   # one entry per row, in order
df.head()

## Cell 6 — Second annotator + Cohen's kappa

A second annotator (a fellow on the AI Office team) re-labeled the same 30 outputs. They disagree on roughly 6 cases — exactly the situation where kappa is informative.

**TODO:** call `cohen_kappa_score` and interpret. Rough scale: <0.2 poor, 0.2-0.4 fair, 0.4-0.6 moderate, 0.6-0.8 substantial, >0.8 almost perfect.

In [ ]:
from sklearn.metrics import cohen_kappa_score

# Synthetic second-annotator labels (same length as seed_cases). Provided for the lab.
annotator2 = [
    'correct', 'correct', 'partially_correct', 'correct', 'partially_correct',
    'incorrect_and_harmful', 'correct', 'correct', 'partially_correct', 'correct',
    # TODO: extend to len(seed_cases) once you reach 30
]

# TODO: kappa = cohen_kappa_score(df['human_label'], annotator2)
# TODO: print kappa and write a 1-sentence interpretation.
...

## Cell 7 — Inspect 3 disagreements

**TODO:** print case + both labels + model output for 3 rows where the two labelers disagree. Write one sentence per row explaining *why* a reasonable clinician could go either way.

In [ ]:
# TODO: disagreements = df[df['human_label'] != annotator2[:len(df)]]
# TODO: for _, row in disagreements.head(3).iterrows(): print(...)
...

## Cell 8 — Refine the rubric (v2)

Based on the disagreements above, **TODO: edit this markdown cell** to add explicit decision rules. Examples to consider:

- *Is "chemotherapy and radiation as indicated" partial or correct?* (decide: partial — must name the regimen)
- *Is omitting tumor weight in a Stage I infant case harmful?* (decide: yes — tumor weight <550 g changes the regimen)
- *Is naming "DD4A" instead of "regimen M" a partial?* (decide: correct — they are equivalent COG names)

Write 3 explicit rules below.

## Cell 9 — Re-label the 6 disagreement cases and recompute kappa

**TODO:** apply the v2 rubric to the disagreements only, then recompute kappa over all 30. Expect kappa to rise.

In [ ]:
# TODO: human_label_v2 = df['human_label'].copy()
# TODO: update the disagreement rows with the v2 decision
# TODO: kappa_v2 = cohen_kappa_score(human_label_v2, annotator2)
# TODO: print before/after kappa
...

## Cell 10 — Save versioned dataset

Lock v1 to disk. The rubric, the labels, and the agreement metric all travel with the data. Future labs will load this file.

In [ ]:
dataset_v1 = {
    'version': 'v1',
    'rubric': {
        'correct': 'Faithful, complete, recommended next step matches COG/SIOP standard.',
        'partially_correct': 'Faithful but next step is generic or a section is missing.',
        'incorrect_and_harmful': 'Invented fact, wrong regimen, or misclassified histology.',
    },
    'cases': [],   # TODO: fill from df  → list of {'id', 'case', 'output', 'human_label'}
    'inter_annotator_kappa': None,   # TODO: kappa_v2
    'n_cases': len(seed_cases),
}
# TODO: with open('dataset_v1.json', 'w') as f: json.dump(dataset_v1, f, indent=2)
...

## Stretch — Targeted synthetic inputs

Look at the cases labeled `partially_correct` or `incorrect_and_harmful`. What pattern do they share? (e.g., the assistant struggles whenever tumor rupture is in the input.)

**TODO:** write 5 more synthetic inputs that target that pattern, run them through the assistant, label them, and append to `dataset_v1`.

In [ ]:
stretch_cases = [
    # TODO: 5 inputs targeting the failure pattern you observed
]
...

## KHCC Connection

At KHCC, the AI Office cannot deploy a clinical assistant on the strength of a demo. A versioned eval dataset with a locked rubric and an inter-annotator kappa is what lets us say *"this assistant performs at X% on cases that look like our patients"* — and what lets us prove the next prompt iteration is actually better, not just different. The dataset you built today is the artifact every future improvement will be measured against.